In [16]:
import os
from pathlib import Path

import gspread
import pandas as pd
from google.oauth2.service_account import Credentials
from merge_tables.db.connection import connect_to_postgres_via_duckdb

duck = connect_to_postgres_via_duckdb()

# From URL: https://docs.google.com/spreadsheets/d/<SPREADSHEET_ID>/edit
SPREADSHEET_ID = "1crYm3XHYZ63SkhFYC_Lk8m-mFjMUYZB4BQ31I6yascE"
SHEET_NAME = "Berlin"

# credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
credentials_path = "./config/gsheet-creds.json"
# Or set explicitly: credentials_path = Path.home() / "secrets" / "service-account.json"
if not credentials_path:
    raise FileNotFoundError(
        "Set GOOGLE_APPLICATION_CREDENTIALS to your service account JSON path"
    )
credentials_path = Path(credentials_path).expanduser()

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'


In [17]:
import duckdb

# Run the first gspread config cell so `credentials_path`, `SPREADSHEET_ID`, and `SHEET_NAME` exist.


def _sql_literal(s: str) -> str:
    return s.replace("'", "''")


key_path = _sql_literal(str(credentials_path.resolve()))

duck.execute("INSTALL gsheets FROM community;")
duck.execute("LOAD gsheets;")
duck.execute(
    f"""
CREATE OR REPLACE SECRET gsheet_sa (
    TYPE gsheet,
    PROVIDER key_file,
    FILEPATH '{key_path}'
);
"""
)

In [18]:
sheets = [
    'Berlin', 'Hamburg', 'Düsseldorf', 'Frankfurt', 'Kiel', 'Köln', 'München', 'Rostock', 'Stuttgart'
]

In [22]:
employer_user_dfs = []

In [23]:
for sheet in sheets:
    df = duck.sql(
        f"""
    SELECT * FROM read_gsheet(
        '{_sql_literal(SPREADSHEET_ID)}',
        sheet='{sheet}',
        all_varchar=true,
        range='A3:J'
    )
    """
    ).df()
    employer_user_dfs.append(df)
    print(f'{sheet} : {df.shape}')

Berlin : (292, 10)
Hamburg : (181, 10)
Düsseldorf : (125, 10)
Frankfurt : (101, 10)
Kiel : (1, 10)
Köln : (114, 10)
München : (78, 10)
Rostock : (94, 10)
Stuttgart : (81, 10)


In [25]:
employer_user_df = pd.concat(employer_user_dfs)

In [27]:
employer_user_df

,BAS Standort,Easybill ID*,Firmenname,Vor- und Nachname des Ansprechpartners*,Geschlecht*,Art des Ansprechpartners*,E-Mail-Addresse*,Telefonnummer,Jobbeschreibung*,Profil in padoa*\n(für mehr Info siehe Tab padoa Profile)
0,Berlin,1130100311,&MICA GmbH (ehem. Michels Architekturbüro) - B...,Hannah Termünde,F,Intern,termuende@undmica.de,NaN,"Officemanagement, Ass. der GF",Administrator
1,Berlin,127000001,1000hands AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Berlin,130000812,3S Antriebe GmbH,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Berlin,130001511,4Panels GmbH & Co. KG,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Berlin,100020015,ABK Allgemeine Beamten Bank AG,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
76,Stuttgart,123010018,WS Wärmeprozesstechnik GmbH,Roman Mothes,M,Intern,r.mothes@flox.com,07159 1632 15,Beauftragter für Gebäudemanagement und Arbeits...,NaN
77,Stuttgart,1300009576,X1F // X1F GmbH - Betriebsstätten Stuttgart (2),NaN,NaN,NaN,NaN,NaN,NaN,NaN
78,Stuttgart,1300016977,X1F // X1F Management & Technology Services Gm...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
79,Stuttgart,1300016965,X1F // X1F Products GmbH (ehem. IKOR-Ideen) - ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
duck.register("employer_user_df", employer_user_df.astype(object))
duck.sql("select * from employer_user_df")


┌──────────────┬──────────────┬──────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────┬─────────────┬───────────────────────────┬───────────────────────────────────┬────────────────────┬──────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────┐
│ BAS Standort │ Easybill ID* │                                    Firmenname                                    │ Vor- und Nachname des Ansprechpartners* │ Geschlecht* │ Art des Ansprechpartners* │         E-Mail-Addresse*          │   Telefonnummer    │                     Jobbeschreibung*                     │ Profil in padoa*\n(für mehr Info siehe Tab padoa Profile) │
│   varchar    │   varchar    │                                     varchar                                      │                 varchar                 │   varchar   │          varchar          │              varchar              │      varchar   

In [35]:
duck.sql(
    """
    create or replace table pg.bas_firms.employer_users as
    select 
        "BAS Standort" as standort,
        "Easybill ID*" as easybill_id,
        Firmenname as firm_name,
        "Vor- und Nachname des Ansprechpartners*" as full_name,
        "Geschlecht*" as sexe,
        "Art des Ansprechpartners*" as person_type,
        "E-Mail-Addresse*" as email,
        "Telefonnummer" as phone,
        "Jobbeschreibung*" as job,
        "Profil in padoa*\n(für mehr Info siehe Tab padoa Profile)" as padoa_profil
    from employer_user_df
    """
)

In [60]:
duck.sql(
    """
    insert into pg.easybill.documents
    by name
    select * from read_csv(
        '/Users/adrienblanquer/Downloads/Documents-Export-29_04_2026-10_56_58.csv',
        types={'Kontakt (Aktuell): Postleitzahl': 'VARCHAR'},
        ignore_errors=true
    )
    """
)


In [59]:
# Step 1: list all VARCHAR(50) columns on pg.easybill.documents and print the ALTER statements.
# Review the output, then run the next cell to actually apply the changes.

varchar50_cols = duck.sql("""
    SELECT column_name
    FROM postgres_query('pg', $$
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = 'easybill'
          AND table_name = 'documents'
          AND data_type = 'character varying'
          AND character_maximum_length = 50
        ORDER BY ordinal_position
    $$)
""").df()["column_name"].tolist()

print(f"Found {len(varchar50_cols)} VARCHAR(50) columns:\n")
alter_stmts = [
    f'ALTER TABLE easybill.documents ALTER COLUMN "{c}" TYPE VARCHAR;'
    for c in varchar50_cols
]
print("\n".join(alter_stmts))

Found 183 VARCHAR(50) columns:

ALTER TABLE easybill.documents ALTER COLUMN "Dokument: Dokumentnummer" TYPE VARCHAR;
ALTER TABLE easybill.documents ALTER COLUMN "Dokument: Typ" TYPE VARCHAR;
ALTER TABLE easybill.documents ALTER COLUMN "Dokument: Titel" TYPE VARCHAR;
ALTER TABLE easybill.documents ALTER COLUMN "Dokument: Datum" TYPE VARCHAR;
ALTER TABLE easybill.documents ALTER COLUMN "Dokument: Fälligkeitsdatum" TYPE VARCHAR;
ALTER TABLE easybill.documents ALTER COLUMN "Dokument: Nettobetrag" TYPE VARCHAR;
ALTER TABLE easybill.documents ALTER COLUMN "Dokument: Bruttobetrag" TYPE VARCHAR;
ALTER TABLE easybill.documents ALTER COLUMN "Dokument: Ust. Betrag" TYPE VARCHAR;
ALTER TABLE easybill.documents ALTER COLUMN "Dokument: Währung" TYPE VARCHAR;
ALTER TABLE easybill.documents ALTER COLUMN "Dokument: Gezahlter Betrag" TYPE VARCHAR;
ALTER TABLE easybill.documents ALTER COLUMN "Dokument: Zahlungsdatum" TYPE VARCHAR;
ALTER TABLE easybill.documents ALTER COLUMN "Dokument: Status" TYPE VARCHA

In [58]:
# Step 2: apply the ALTERs. Run only after reviewing the list above.
# Wrapped in a transaction so it's all-or-nothing.

if not varchar50_cols:
    print("Nothing to do.")
else:
    duck.execute("BEGIN;")
    try:
        for stmt in alter_stmts:
            duck.execute(stmt)
        duck.execute("COMMIT;")
        print(f"Altered {len(alter_stmts)} columns to VARCHAR (unbounded).")
    except Exception:
        duck.execute("ROLLBACK;")
        raise

BinderException: Binder Error: Unsupported ALTER TABLE type - Postgres tables only support RENAME TABLE, RENAME COLUMN, ADD COLUMN and DROP COLUMN